# Multi-Turn Conversations

**Claude doesn't store any of your conversation history.** Each request is completely independent, with no memory of previous exchanges. If you want a multi-turn conversation where Claude remembers earlier context, *you* have to manage the conversation state.

To maintain context you do two things:

1. Manually maintain a list of all messages in your code
2. Send the **complete** message history with every request

The flow that actually works:

1. Send your initial user message to Claude
2. Take Claude's response and add it to your message list as an **assistant** message
3. Add your follow-up question as another **user** message
4. Send the entire conversation history to Claude

> Make sure the kernel is **Python (anthropic-cert)**. This notebook re-does the setup so it can run on its own.

## Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"


def get_text(message):
    """Return the assistant's text, skipping any thinking blocks.

    The course uses message.content[0].text, but newer models (like
    Sonnet 5) may put a ThinkingBlock before the TextBlock, so
    content[0] isn't always the text. We grab the block whose type
    is 'text' instead of assuming position.
    """
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


print(f"Client ready. Model: {model}")

## The problem with stateless conversations

Say you ask Claude *"What is quantum computing?"* and get a good response. Then in a **brand-new request** you follow up with *"Write another sentence"* — Claude has no idea what you're referring to, because it has no memory of the first exchange. It'll write a sentence about something random.

Let's see that failure first:

In [ ]:
# A second, independent request that does NOT include the earlier context.
orphan = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[{"role": "user", "content": "Write another sentence"}],
)

print(get_text(orphan))   # unrelated to quantum computing — Claude has no context

## Building helper functions

To make conversation management easier, we create three helpers. These come straight from the course and you'll reuse them throughout.

The only change from the course: `chat()` returns `get_text(message)` instead of `message.content[0].text`, so a thinking block never breaks it (see the setup cell).

In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return get_text(message)

## Putting it all together

Now the same follow-up works, because we send the **complete** history. Claude sees the quantum-computing definition and its own reply, so *"Write another sentence"* correctly expands on it.

In [ ]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

# Append the follow-up answer too, so the stored history is complete
add_assistant_message(messages, final_answer)

print("First answer:\n", answer)
print("\nFollow-up (with context):\n", final_answer)

## Inspect the full conversation

See how the list alternates user / assistant / user / assistant. This list *is* the conversation memory.

In [ ]:
for m in messages:
    print(f"[{m['role']}] {m['content']}\n")